In [ ]:
!pip install -q -U transformers chromadb sentence-transformers "pillow<12.0"
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("HuggingFace login successful!")

import torch
import time
from transformers import AutoModel, AutoTokenizer, AutoImageProcessor
from sentence_transformers import SentenceTransformer
import chromadb
from PIL import Image
import os
import shutil
import io

INPUT_DATA_PATH = "/kaggle/input/knowledge-dataset/"
READ_ONLY_DB_PATH = os.path.join(INPUT_DATA_PATH, "embedding_vectors")
IMG_PATH = os.path.join(INPUT_DATA_PATH, "all_images")
WRITABLE_DB_PATH = "/kaggle/working/my_vector_db"

print(f"Copying DB to writable path: {WRITABLE_DB_PATH}")

if os.path.exists(WRITABLE_DB_PATH):
    shutil.rmtree(WRITABLE_DB_PATH)

shutil.copytree(READ_ONLY_DB_PATH, WRITABLE_DB_PATH)
print("Vector DB copied successfully.")

print("Loading InternVL3.5-8B...")

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoImageProcessor.from_pretrained(
    "OpenGVLab/InternVL3_5-8B",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(
    "OpenGVLab/InternVL3_5-8B",
    trust_remote_code=True
)
model = AutoModel.from_pretrained(
    "OpenGVLab/InternVL3_5-8B",
    trust_remote_code=True,
    dtype="auto",
    device_map="auto"
).eval()

print("InternVL3.5-8B loaded.")

embed_model = SentenceTransformer(
    "google/embeddinggemma-300m",
    device="cpu"
)

print("Loading ChromaDB...")
client = chromadb.PersistentClient(path=WRITABLE_DB_PATH)
collection = client.get_collection("assam_disasters_multimodal")
print("DB Loaded. Items:", collection.count())

def analyze_image(image_path, analysis_prompt="Describe this image in detail."):
    print(f"ANALYZING IMAGE: {image_path}")

    try:
        image = Image.open(image_path).convert("RGB")
    except FileNotFoundError:
        print(f"ERROR: Image file not found at {image_path}. Please check the path.")
        return
    except Exception as e:
        print(f"ERROR loading image: {e}")
        return
    proc = processor(images=[image], return_tensors="pt")
    pixel_values = proc["pixel_values"].to(model.device).to(torch.bfloat16)

    gen_cfg = {"max_new_tokens": 512}

    print("Generating description...")
    start_time = time.time()
    response = model.chat(
        tokenizer,
        pixel_values,
        analysis_prompt,
        gen_cfg
    )
    end_time = time.time()
    latency = end_time - start_time

    print(response)
    print(f"Inference Latency: {latency:.4f} seconds")
def run_multimodal_rag(question, n_results=4):
    print(f"QUESTION: {question}")
    q_embed = embed_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[q_embed],
        n_results=n_results
    )

    if not results["documents"][0]:
        print("No relevant results found.")
        return

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]

    context_texts = []
    context_images = []

    for doc, meta in zip(documents, metadatas):
        context_texts.append(doc)
        img_str = meta.get("image_paths", "")
        if img_str:
            for p in img_str.split(","):
                img_name = os.path.basename(p)
                real_path = os.path.join(IMG_PATH, img_name)
                if os.path.exists(real_path):
                    context_images.append(real_path)

    context_images = list(set(context_images))
    print(f"Retrieved {len(context_texts)} text chunks and {len(context_images)} images.")
    loaded_images = []
    for img in context_images[:2]:
        try:
            loaded_images.append(Image.open(img).convert("RGB"))
        except:
            pass

    if loaded_images:
        proc = processor(images=loaded_images, return_tensors="pt")
        pixel_values = proc["pixel_values"].to(model.device).to(torch.bfloat16)
    else:
        pixel_values = None
    gen_cfg = {"max_new_tokens": 256}

    print("Generating answer...")
    start_time = time.time()

    response = model.chat(
        tokenizer,
        pixel_values,
        prompt,
        gen_cfg
    )
    end_time = time.time()
    latency = end_time - start_time

    print("\n=========== ANSWER ===========\n")
    print(response)
    for m in metadatas:
        print(f"- {m['source_pdf']} (page {m['page_number']})")
    print(f"Inference Latency: {latency:.4f} seconds")

def run_analysis():
    print("InternVL Analysis")
    while True:
        choice = input("Select mode (T: Text RAG | I: Image Analysis | Q: Quit): ").upper().strip()

        if choice == 'Q':
            print("Exiting analysis.")
            break

        elif choice == 'T':
            question = input("Enter your RAG query (e.g., 'What caused the landslide?'): ")
            try:
                n_results = int(input("Enter number of documents to retrieve (e.g., 4): "))
            except ValueError:
                n_results = 4
                print("Invalid number, defaulting to 4 results.")

            run_multimodal_rag(question, n_results)

        elif choice == 'I':
            print("\n--- Image Analysis Mode ---")
            image_path = input(f"Enter the FULL path to the image file (e.g., {IMG_PATH}/example.png): ")
            analysis_prompt = input("Enter the analysis prompt (e.g., 'Describe the scene'): ")
            analyze_image(image_path, analysis_prompt)
        else:
            print("Invalid choice. Please enter T, I, or Q.")
run_analysis()